In [8]:
import torch
import torch.nn as nn
from torch.optim import Adam
from torchMPC.mpc import mpc
from torchMPC.mpc import util
from torchMPC.mpc.env_dx import cartpole
import torch.nn.functional as F
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np
import math
import os
import pdb

for i in range(1):
    pass
print('GOGOGO!')

GOGOGO!


In [9]:
# costNet running and terminal cost
class mpcTorchCost(nn.Module):
    def __init__(self, n_state, n_ctrl, tau0):
        super().__init__()
        self.n_state = n_state
        self.n_ctrl = n_ctrl

        # Equilibrium point. 注册为buffer，可以跟着模型一起加载到GPU上
        if isinstance(tau0, torch.Tensor):
            self.register_buffer('tau0', tau0)
        else:
            self.register_buffer('tau0', torch.tensor(tau0))

        self.q = nn.Parameter(torch.tensor([1.0, 1.0, 1.0, 1.0, 1.0]))
        self.f = nn.Parameter(torch.randn(n_state, n_state))

        self.register_buffer('R', torch.eye(n_ctrl) * 0.01)

    def forward(self, tau, terminal=False):
        """
        Args:
            tau: [batch_size, n_state + n_ctrl], concatenated form of state and action
            t: Current timestamp
            T: Total prediction horizon
        Returns:
            cost: [batch_size] 
        """
        batch_size = tau.size(0)
        
        # 对于车杆问题，期望渐进稳定到顶点，所以需要减去参考点
        # 控制的参考点就是0
        state = tau[:, :self.n_state] - self.tau0[:self.n_state] 
        ctrl = tau[:, self.n_state:]
        
        # 如果是最后一个时间步，只计算终端损失
        if terminal:
            F_postive = (self.f.T @ self.f).repeat(batch_size, 1, 1)
            return 0.5 * torch.bmm(torch.bmm(state.unsqueeze(1), F_postive), state.unsqueeze(2)).squeeze(-1).squeeze(-1) # Return shape: (batch_size, )
        else:
            return self.running_cost(state, ctrl)
    
    def running_cost(self, state, ctrl): 
        # 简单二次型运行代价
        batch_size = state.shape[0]

        Q_postive = torch.diag(self.q).pow(2).repeat(batch_size, 1, 1)
        xTQx = 0.5 * torch.bmm(torch.bmm(state.unsqueeze(1), Q_postive), state.unsqueeze(2))
        uTRu = 0.5 * torch.bmm(torch.bmm(ctrl.unsqueeze(1), self.R.repeat(batch_size, 1, 1)), ctrl.unsqueeze(2))
        return (xTQx + uTRu).squeeze(-1).squeeze(-1) # Return shape: (batch_size,)

In [10]:
# 均匀分布采样
def uniform(shape, low, high):
    r = high - low
    return torch.rand(shape) * r + low

# 生成初始状态，来自mpc torch
def cartpole_initx(n_batch, angle=180):
    ratio = angle / 180.0
    th = uniform(n_batch, -ratio*np.pi, ratio*np.pi)
    thdot = uniform(n_batch, -.5 * ratio, .5 * ratio)
    x = uniform(n_batch, -0.5 * ratio, 0.5 * ratio)
    xdot = uniform(n_batch, -0.5 * ratio, 0.5 * ratio)
    xinit = torch.stack((x, xdot, torch.cos(th), torch.sin(th), thdot), dim=1)
    ref = torch.tensor([0., 0., 1., 0., 0.])
    return xinit 

# 可视化

In [4]:
device = 'cpu'

n_batch, T, mpc_T = 32, 100, 32

dx = cartpole.CartpoleDx()
t_dir = "D:/Docs/code_lib/graduation_test/cartpole_pic"

equilibrium = dx.goal_state.to(device)
cost = mpcTorchCost(dx.n_state, dx.n_ctrl, equilibrium).to(device)

model_path = 'D:/Docs/code_lib/graduation_test/testCartPole/250514Train_GO2!/pth/64.pth'
cost.load_state_dict(torch.load(model_path))

x = cartpole_initx(n_batch)
u_init = None
for t in tqdm(range(T)):
    nominal_states, nominal_actions, nominal_objs = mpc.MPC(
        dx.n_state, dx.n_ctrl, mpc_T,
        n_batch=n_batch,
        u_init=u_init,
        # u_lower=dx.lower, u_upper=dx.upper,
        lqr_iter=50,
        verbose=0,
        exit_unconverged=False,
        detach_unconverged=False,
        linesearch_decay=dx.linesearch_decay,
        max_linesearch_iter=dx.max_linesearch_iter,
        grad_method=mpc.GradMethods.AUTO_DIFF,
        eps=1e-2,
    )(x, cost, dx)

    next_action = nominal_actions[0]
    u_init = torch.cat((nominal_actions[1:], torch.zeros(1, n_batch, dx.n_ctrl)), dim=0) # u_init shape (mpc_T, batch_size, 1)
    u_init[-2] = u_init[-3] # 因为u[-1]本来就是0了

    x = dx(x, next_action) # 往前走一步

    # 以下都是用来可视化的
    n_col = 4
    n_row = n_batch // n_col
    fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
    axs = axs.reshape(-1)
    for i in range(n_batch):
        dx.get_frame(x[i], ax=axs[i])
        axs[i].get_xaxis().set_visible(False)
        axs[i].get_yaxis().set_visible(False)
    fig.tight_layout()
    fig.savefig(os.path.join(t_dir, 'frame_{:03d}.png'.format(t)))
    plt.close(fig)

C:\Users\90534\AppData\Local\Temp\ipykernel_27788\497930705.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cost.load_state_dict(torch.load(model_path))
100%|██████████

# 条件判断稳定

In [11]:
def stable_num(state, u, eps=1e-2):
    batch_size = state.shape[0]
    refx = torch.tensor([0., 0., 1., 0., 0.]).unsqueeze(0).repeat(batch_size, 1)
    refu = torch.tensor([0.]).unsqueeze(0).repeat(batch_size, 1)
    whether_stable = torch.all((state - refx) < eps, dim=1) * torch.all((u - refu) < eps, dim=1)
    return torch.sum(whether_stable).item()

# test_states = cartpole_initx(10)
# test_states[0] = torch.tensor([0., 0., 1., 0., 0.])
# test_states[1] = torch.tensor([0., 0., 1., 0., 0.])
# test_states[2] = torch.tensor([0., 0., 1., 0., 0.])
# test_controls = torch.randn(10, 1) * 0.1
# test_controls[0] = torch.tensor([0.])
# test_controls[1] = torch.tensor([0.])
# test_controls[2] = torch.tensor([0.])
# print(stable_num(test_states, test_controls))

In [17]:
device = 'cpu'

n_batch, T, mpc_T = 128, 150, 32

dx = cartpole.CartpoleDx()
t_dir = "D:/Docs/code_lib/graduation_test/cartpole_pic"

equilibrium = dx.goal_state.to(device)
cost = mpcTorchCost(dx.n_state, dx.n_ctrl, equilibrium).to(device)

model_path = 'D:/Docs/code_lib/graduation_test/testCartPole/250514Train_GO2!/pth/64.pth'
cost.load_state_dict(torch.load(model_path))

x = cartpole_initx(n_batch)
u_init = None
x_list = []
u_list = []
for t in tqdm(range(T)):
    nominal_states, nominal_actions, nominal_objs = mpc.MPC(
        dx.n_state, dx.n_ctrl, mpc_T,
        n_batch=n_batch,
        u_init=u_init,
        # u_lower=dx.lower, u_upper=dx.upper,
        lqr_iter=50,
        verbose=0,
        exit_unconverged=False,
        detach_unconverged=False,
        linesearch_decay=dx.linesearch_decay,
        max_linesearch_iter=dx.max_linesearch_iter,
        grad_method=mpc.GradMethods.AUTO_DIFF,
        eps=1e-2,
    )(x, cost, dx)

    next_action = nominal_actions[0]
    x_list.append(x)
    u_list.append(next_action)
    u_init = torch.cat((nominal_actions[1:], torch.zeros(1, n_batch, dx.n_ctrl)), dim=0) # u_init shape (mpc_T, batch_size, 1)
    u_init[-2] = u_init[-3] # 因为u[-1]本来就是0了

    x = dx(x, next_action) # 往前走一步

    # 以下都是用来可视化的
    # n_col = 4
    # n_row = n_batch // n_col
    # fig, axs = plt.subplots(n_row, n_col, figsize=(3*n_col,3*n_row), gridspec_kw = {'wspace':0, 'hspace':0})
    # axs = axs.reshape(-1)
    # for i in range(n_batch):
    #     dx.get_frame(x[i], ax=axs[i])
    #     axs[i].get_xaxis().set_visible(False)
    #     axs[i].get_yaxis().set_visible(False)
    # fig.tight_layout()
    # fig.savefig(os.path.join(t_dir, 'frame_{:03d}.png'.format(t)))
    # plt.close(fig)

C:\Users\90534\AppData\Local\Temp\ipykernel_5136\1537374295.py:12: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  cost.load_state_dict(torch.load(model_path))
100%|██████████

In [18]:
print(x_list[-1])
print(stable_num(x_list[-1], u_list[-1], eps=1e-2))

tensor([[ 2.9108e-05, -1.4003e-05,  1.0000e+00,  6.9846e-07, -3.3726e-07],
        [ 6.7410e-04, -3.2431e-04,  1.0000e+00,  1.6180e-05, -7.7945e-06],
        [ 1.2091e-02, -5.8168e-03,  1.0000e+00,  2.9011e-04, -1.3956e-04],
        [ 1.8440e-02, -8.8708e-03,  1.0000e+00,  4.4242e-04, -2.1284e-04],
        [ 5.0065e-03, -2.4086e-03,  1.0000e+00,  1.2013e-04, -5.7789e-05],
        [ 1.0604e-02, -5.1012e-03,  1.0000e+00,  2.5442e-04, -1.2241e-04],
        [-1.0948e-02,  5.2671e-03,  1.0000e+00, -2.6274e-04,  1.2645e-04],
        [ 2.2376e-02, -1.0764e-02,  1.0000e+00,  5.3682e-04, -2.5820e-04],
        [ 6.3243e-01, -2.0842e-02, -9.9967e-01, -2.5552e-02,  8.6481e-02],
        [ 1.4601e-02, -7.0240e-03,  1.0000e+00,  3.5030e-04, -1.6850e-04],
        [ 6.8683e-03, -3.3042e-03,  1.0000e+00,  1.6480e-04, -7.9271e-05],
        [ 1.1108e-02, -5.3438e-03,  1.0000e+00,  2.6651e-04, -1.2820e-04],
        [-2.7967e-03,  1.3455e-03,  1.0000e+00, -6.7119e-05,  3.2312e-05],
        [-1.1227e-02,  5.

In [41]:
print(x_list[-1])
print(stable_num(x_list[-1], u_list[-1], eps=1e-2))

tensor([[ 3.7981e-03, -1.8279e-03,  1.0000e+00,  9.1536e-05, -4.4922e-05],
        [-2.3775e-03,  1.1431e-03,  1.0000e+00, -5.6589e-05,  2.6215e-05],
        [-2.4633e-03,  1.1844e-03,  1.0000e+00, -5.8702e-05,  2.7354e-05],
        ...,
        [-5.1774e-03,  2.4918e-03,  1.0000e+00, -1.2481e-04,  6.1312e-05],
        [ 2.4145e-03, -1.1627e-03,  1.0000e+00,  5.8631e-05, -2.9733e-05],
        [ 1.6121e-03, -7.7477e-04,  1.0000e+00,  3.8201e-05, -1.7324e-05]],
       grad_fn=<StackBackward0>)
247
